# News Classification Model Training
Training DistilBERT for Informative vs Misinformative news classification

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, TrainingArguments, Trainer
from datasets import Dataset
import evaluate
import numpy as np
import torch
import pandas as pd
import os
import json
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

In [ ]:
MODEL_CKPT = os.getenv("MODEL_CKPT", "distilbert/distilbert-base-uncased")
os.makedirs("../model", exist_ok=True)

label2id = {'Informative': 0, 'Misinformative': 1}
id2label = {0: 'Informative', 1: 'Misinformative'}

num_labels = len(label2id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Helper Functions
def get_data(file_path):
    df = pd.read_csv(file_path)
    return df

def tokenize_function(examples):
    return tokenizer(examples["title"], padding="max_length", truncation=True, max_length=512)

def compute_metrics(eval_pred):
    metric = evaluate.load("accuracy")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# Load Configuration
with open("../config.json", "r") as f:
    all_configs = json.load(f)

args_keys = [key for key in all_configs.keys() if key.startswith("args")]
print(f"Found {len(args_keys)} configurations: {args_keys}")

In [ ]:
# Track results
all_results = []

for args_key in args_keys:
    print(f"\n{'='*80}")
    print(f"Training with {args_key}")
    print(f"{'='*80}\n")
    
    config_args = all_configs[args_key]
    print(f"Config: {config_args}\n")
    
    config = AutoConfig.from_pretrained(
        MODEL_CKPT,
        label2id=label2id,
        id2label=id2label,
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_CKPT, config=config).to(device)

    train_df = get_data("../results/train.csv")
    val_df = get_data("../results/val.csv")

    print(train_df.columns)
    print(train_df.head())

    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)

    train_dataset = train_dataset.map(tokenize_function, batched=True)
    val_dataset = val_dataset.map(tokenize_function, batched=True)

    train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

    training_args = TrainingArguments(
    output_dir=config_args["output_dir"],
    num_train_epochs=config_args["num_train_epochs"],
    learning_rate=config_args["learning_rate"],
    per_device_train_batch_size=config_args["per_device_train_batch_size"],
    per_device_eval_batch_size=config_args["per_device_eval_batch_size"],
    weight_decay=config_args["weight_decay"],
    disable_tqdm=config_args["disable_tqdm"],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

    trainer = Trainer(
        model=model,
        args=training_args,
        compute_metrics=compute_metrics, 
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    print(model.config)

    train_result = trainer.train()
    eval_result = trainer.evaluate()
    
    print(f"\n{args_key} Results:")
    print(f"  Training Loss: {train_result.training_loss:.4f}")
    print(f"  Validation Accuracy: {eval_result['eval_accuracy']:.4f}")
    
    result = {
        "config_name": args_key,
        "hyperparameters": config_args,
        "train_loss": train_result.training_loss,
        "val_accuracy": eval_result['eval_accuracy'],
        "eval_loss": eval_result['eval_loss']
    }
    all_results.append(result)
    
    result_file = os.path.join(config_args["output_dir"], "training_results.json")
    with open(result_file, "w") as f:
        json.dump(result, f, indent=2)
    
    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# Summary
print("\n" + "="*80)
print("TRAINING COMPLETE - SUMMARY")
print("="*80)

for result in sorted(all_results, key=lambda x: x['val_accuracy'], reverse=True):
    print(f"\n{result['config_name']}:")
    print(f"  Val Accuracy: {result['val_accuracy']:.4f}")
    print(f"  Train Loss: {result['train_loss']:.4f}")
    print(f"  Hyperparameters: {result['hyperparameters']}")

best_result = max(all_results, key=lambda x: x['val_accuracy'])
print("\n" + "="*80)
print("BEST CONFIGURATION")
print("="*80)
print(f"Config: {best_result['config_name']}")
print(f"Validation Accuracy: {best_result['val_accuracy']:.4f}")
print(f"Hyperparameters: {best_result['hyperparameters']}")

with open("../model/all_results_summary.json", "w") as f:
    json.dump({
        "all_results": all_results,
        "best_config": best_result
    }, f, indent=2)

print(f"\nSummary saved to ../model/all_results_summary.json")